In [3]:

!pip install -q ultralytics onnx onnxruntime-gpu


In [1]:

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Please enable GPU in Runtime settings")


PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4


In [2]:

!pip install -q ultralytics onnx onnxruntime-gpu


In [4]:

import os
import torch
import onnx
import yaml
import shutil

from ultralytics import YOLO

print("All libraries imported successfully!")


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
All libraries imported successfully!


In [5]:

# Load YOLOv8 Nano pretrained model
model = YOLO("yolov8n.pt")

# Display model architecture
model.info()

print("Model architecture loaded!")


YOLOv8n summary: 129 layers, 3,157,200 parameters, 0 gradients, 8.9 GFLOPs
Model architecture loaded!


In [6]:

os.makedirs("/content/skill7_outputs", exist_ok=True)

# Save architecture configuration
with open("/content/skill7_outputs/model_architecture.yaml", "w") as f:
    yaml.safe_dump(model.model.yaml, f)

print("Architecture configuration saved!")


Architecture configuration saved!


In [7]:

# Save PyTorch checkpoint
model.save("/content/skill7_outputs/best.pt")

print("Checkpoint saved successfully!")


Checkpoint saved successfully!


In [8]:

# Export model to ONNX
onnx_path = model.export(
    format="onnx",
    imgsz=640,
    opset=12,
    simplify=True
)

print("ONNX exported:", onnx_path)


Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs

PyTorch: starting from 'yolov8n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (6.2 MB)
requirements: Ultralytics requirement ['onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.13.16 environment at: /usr
Resolved 10 packages in 238ms
Prepared 2 packages in 32ms
Installed 2 packages in 5ms
 + colorama==0.4.6
 + onnxslim==0.1.99

requirements: AutoUpdate success ✅ 0.6s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.23.2 opset 12...
ONNX: slimming with onnxslim 0.1.99...
ONNX: export success ✅ 4.7s, saved as 'yolov8n.onnx' (12.3 MB)

Export complete (5.1s)
Resu

In [9]:

# Load ONNX model
onnx_model = onnx.load(onnx_path)

# Check graph validity
onnx.checker.check_model(onnx_model)

print("ONNX model is valid!")
print("Graph inputs:", len(onnx_model.graph.input))
print("Graph outputs:", len(onnx_model.graph.output))


ONNX model is valid!
Graph inputs: 1
Graph outputs: 1


In [10]:

# Export TensorRT engine using FP16 precision
engine_path = model.export(
    format="engine",
    imgsz=640,
    half=True,
    device=0
)

print("TensorRT engine:", engine_path)


WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8n summary (fused): 72 layers, 3,151,904 parameters, 0 gradients, 8.7 GFLOPs

PyTorch: starting from 'yolov8n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (6.2 MB)

ONNX: starting export with onnx 1.23.2 opset 18...
ONNX: slimming with onnxslim 0.1.99...
ONNX: export success ✅ 0.9s, saved as 'yolov8n.onnx' (12.3 MB)
requirements: Ultralytics requirement ['tensorrt-cu13>=7.0.0,!=10.2.0'] not found, attempting AutoUpdate...
Using Python 3.13.16 environment at: /usr
Resolved 3 packages in 147ms
Prepared 3 packages in 1m 28s
Installed 3 packages in 6ms
 + tensorrt-cu13==11.5.0.63
 + tensorrt-cu13-bindings==11.5.0.63
 + tensorrt-cu13-libs==11.5.0.63

requirements: AutoUpdate success ✅ 89.1s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


Tensor

2026-11-23 06:46:23,640 - [modelopt][onnx] - INFO - Successfully enabled 1 EPs for ORT: ['CPUExecutionProvider']


INFO:modelopt.onnx:Successfully enabled 1 EPs for ORT: ['CPUExecutionProvider']


2026-11-23 06:46:24,059 - [modelopt][onnx] - INFO - Running ONNX Runtime to obtain reference outputs (this may take a while)...


INFO:modelopt.onnx.autocast:Running ONNX Runtime to obtain reference outputs (this may take a while)...


2026-11-23 06:46:24,828 - [modelopt][onnx] - WARNING - Did not find  in value info map! Assuming not castable


2026-11-23 06:46:25,798 - [modelopt][onnx] - INFO - Converted 233/233 nodes (100.00%) to fp16


INFO:modelopt.onnx.autocast:Converted 233/233 nodes (100.00%) to fp16


TensorRT: input "images" with shape(1, 3, 640, 640) DataType.FLOAT
TensorRT: output "output0" with shape(1, 84, 8400) DataType.FLOAT
TensorRT: building FP16 engine as yolov8n.engine
TensorRT: export success ✅ 245.6s, saved as 'yolov8n.engine' (62.2 MB)

Export complete (247.5s)
Results saved to /content/yolov8n.engine
Predict:         yolo predict task=detect model=yolov8n.engine imgsz=640 quantize=16
Validate:        yolo val task=detect model=yolov8n.engine imgsz=640 data=coco.yaml quantize=16
Visualize:       https://netron.app
TensorRT engine: yolov8n.engine


In [11]:

# Load TensorRT engine
trt_model = YOLO(engine_path)

# Test on a sample image
results = trt_model.predict(
    source="https://ultralytics.com/images/bus.jpg",
    conf=0.25,
    save=True
)

print("TensorRT inference completed!")


Loading yolov8n.engine for TensorRT inference...

image 1/1 /content/bus.jpg: 640x640 4 persons, 1 bus, 3.6ms
Speed: 28.2ms preprocess, 3.6ms inference, 26.3ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /content/runs/detect/predict
TensorRT inference completed!


In [12]:

output_dir = "/content/skill7_outputs"

print("PROJECT ARTEFACTS")

for file in os.listdir(output_dir):
    path = os.path.join(output_dir, file)
    print(file, "->", round(os.path.getsize(path)/1024, 2), "KB")


PROJECT ARTEFACTS
best.pt -> 6329.6 KB
model_architecture.yaml -> 1.06 KB


In [13]:
!zip -r /content/Skill7_Outputs.zip /content/skill7_outputs

  adding: content/skill7_outputs/ (stored 0%)
  adding: content/skill7_outputs/best.pt (deflated 8%)
  adding: content/skill7_outputs/model_architecture.yaml (deflated 78%)


In [14]:
from google.colab import files
files.download("/content/Skill7_Outputs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>